# Unit 5 exercises: matrix factorization -- latent factors, the logistic loss, and gradient descent

Run the recsys data generators before you begin. These exercises build the latent-factor path from the ground up. First you meet the model itself -- every reader and every book is a short vector of hidden **latent factors**, and the score for a pair is their plain **dot product**. Then you build the **logistic implicit-feedback loss** over the train positives plus complement-sampled negatives, take **full-batch, per-entity-averaged gradient-descent** steps by hand on a tiny two-cluster toy log, and watch the loss fall. Only then do you reveal `bookrec.MatrixFactorizationPath`, fit it on the real log at the pinned config, and score it on the Unit 1 validation scoreboard against the random floor, popularity, the Unit-3 lexical path, and Unit 4's item-item CF -- where MF lands **on par with CF** (within noise), not beating it. The two Challenge exercises explore how many factors the model needs and show that MF's score is *literally* an embedding dot product -- the bridge to Unit 8's neural two-tower. A reader's `seen` set is always their train-positive history; `val` is the sealed holdout we grade against. Each answer is left for you to complete; no solutions live here.

In [ ]:
import json

import bookrec
import numpy as np
import pandas as pd

SEED = 0  # seed every toy experiment so the loss curve and the learned factors are reproducible

# --- the real interaction log (the same substrate as Units 1-4) ----------------------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])

# Each reader's train-positive history is their `seen` set (the exclusion set at retrieval).
train = interactions[interactions["split"] == "train"]
train_positive = train[train["label"] == 1]
seen_by = (
    train_positive.groupby("reader_id")["item_id"]
    .apply(lambda items: {int(i) for i in items})
    .to_dict()
)
col_of = {item_id: col for col, item_id in enumerate(catalog_ids)}

# The pinned matrix-factorization config, measured on this data. Do NOT shrink the epochs:
# below ~300 the model silently drops under the lexical path.
PINNED = {
    "n_factors": 32,
    "n_epochs": 300,
    "learning_rate": 0.5,
    "reg": 0.05,
    "n_negatives": 10,
    "seed": SEED,
}

# --- a tiny, hand-checkable toy log: two taste clusters ------------------------------------
# Readers 0,1,2 each read books 10,11,12; readers 3,4,5 each read books 20,21,22. Six readers,
# six books -- small enough to trace the factors and the loss by hand.
TOY_CATALOG = [10, 11, 12, 20, 21, 22]
TOY_PAIRS = [(r, b) for r in (0, 1, 2) for b in (10, 11, 12)]
TOY_PAIRS += [(r, b) for r in (3, 4, 5) for b in (20, 21, 22)]
TOY_ROWS = [
    {"reader_id": r, "item_id": b, "split": "train", "label": 1} for r, b in TOY_PAIRS
]
toy_book_col = {b: j for j, b in enumerate(TOY_CATALOG)}
n_readers_toy = 6
n_books_toy = len(TOY_CATALOG)


def sigmoid(z):
    """The logistic function 1 / (1 + e^-z): maps a raw score to a (0, 1) like-probability."""
    return 1.0 / (1.0 + np.exp(-z))


def accumulate(index, weighted, size):
    """Per-entity gradient sum (GIVEN helper): row e of the result is the sum of `weighted`
    over every training pair whose entity index is e -- a per-column np.bincount, the fast
    per-entity gradient path the shipped path uses."""
    out = np.zeros((size, weighted.shape[1]))
    for d in range(weighted.shape[1]):
        out[:, d] = np.bincount(index, weights=weighted[:, d], minlength=size)
    return out

## Exercise 1 - The latent-factor model: a score is a dot product of hidden taste factors

Matrix factorization describes every reader and every book by a short vector of hidden **latent factors** -- a handful of "taste dials". Reader $u$ is a row $P_u$ of the reader-factor matrix $P$ (readers x factors); book $i$ is a row $Q_i$ of the item-factor matrix $Q$ (books x factors). The model's raw score for the pair is their **dot product** $P_u \cdot Q_i$, and the predicted like-probability is $\sigma(P_u \cdot Q_i)$ -- a large positive dot product means the reader's taste dials line up with the book's, so a likely read.

Below are tiny hand-made factors for 2 readers and 3 books, with 2 latent factors each. Form the full score matrix $P Q^{\top}$ (readers x books) with a single matrix product, squash it to like-probabilities with `sigmoid`, and read back reader 0's highest-scoring book. In two or three sentences, say what a factor "means" here (it is learned, not labelled) and why the score being a plain dot product is the fact this whole unit -- and Unit 8's two-tower -- rests on.

In [ ]:
P_demo = np.array([[1.5, -0.4], [-1.1, 1.3]])             # 2 readers x 2 latent factors
Q_demo = np.array([[1.4, -0.3], [-0.9, 1.2], [0.2, 0.1]])  # 3 books x 2 latent factors

score_matrix = None       # TODO: P_demo @ Q_demo.T  -- (readers x books); entry [u, i] = P_u . Q_i
prob_matrix = None        # TODO: sigmoid(score_matrix) -- predicted like-probability in (0, 1)
reader0_scores = None     # TODO: score_matrix[0] -- reader 0's raw score for each of the 3 books
reader0_top_book = None   # TODO: int index (0, 1, or 2) of reader 0's highest-scoring book,
                          #       e.g. int(np.argsort(reader0_scores)[-1])
factors_explanation = "TODO"  # 2-3 sentences: a latent factor is a LEARNED, unlabelled taste
                              # dimension; the score is a plain dot product of the two factor
                              # vectors -- the exact shape Unit 8's two-tower will learn

## Exercise 2 - Implicit feedback: positives, sampled negatives, and the logistic loss

Our feedback is **implicit**: a train positive (`label == 1`) says a reader engaged a book, but there are no negative *ratings*. To learn, we pair each observed positive with `n_negatives` **sampled negatives** drawn from that reader's **unobserved complement** -- random books the reader has *not* read (resampling any draw that happens to hit one of their own positives). These complement negatives, **not** the log's `label == 0` exposure rows (which carry taste signal and would wreck training -- the lesson measures both), are the negatives that work.

With labels $y \in \{0, 1\}$ and model score $s = P_u \cdot Q_i$, the **logistic implicit-feedback loss** of one pair is $-[\,y \log \sigma(s) + (1 - y)\log(1 - \sigma(s))\,]$, averaged over all pairs; driving it down pushes $\sigma$ toward $1$ on positives and toward $0$ on sampled negatives. On the toy log, build the positive index arrays, sample `n_negatives = 10` complement negatives **per positive**, stack the positive-then-negative `u_idx` / `i_idx` / `y` arrays, and write `mean_logistic_loss(P, Q, u_idx, i_idx, y)`. Report the per-positive negative count and the loss of a fresh random $P, Q$ (expect roughly $\log 2 \approx 0.69$ at tiny random init, since $\sigma(0) = 0.5$).

In [ ]:
rng = np.random.default_rng(SEED)
n_negatives = 10

# Positive pairs as entity-index arrays (a toy reader id already is its index; map books to cols).
pos_u = None              # TODO: np.array([r for r, b in TOY_PAIRS])
pos_i = None              # TODO: np.array([toy_book_col[b] for r, b in TOY_PAIRS])
n_pos = None              # TODO: len(TOY_PAIRS)

# Sample n_negatives complement negatives PER positive: repeat each positive reader n_negatives
# times, draw random books, and resample any (reader, book) that collides with an observed pair.
observed = None           # TODO: the set of observed (reader_index, book_column) pairs
neg_u = None              # TODO: np.repeat(pos_u, n_negatives)
neg_i = None              # TODO: rng.integers(0, n_books_toy, size=n_pos * n_negatives)
# TODO: while some (neg_u[j], neg_i[j]) is in `observed`, redraw those neg_i[j] with rng.integers

u_idx = None              # TODO: np.concatenate([pos_u, neg_u])
i_idx = None              # TODO: np.concatenate([pos_i, neg_i])
y = None                  # TODO: np.concatenate([np.ones(n_pos), np.zeros(n_pos * n_negatives)])


def mean_logistic_loss(P, Q, u_idx, i_idx, y):
    # TODO: scores = (P[u_idx] * Q[i_idx]).sum(axis=1); p = sigmoid(scores);
    #       return the mean of -(y * log(p) + (1 - y) * log(1 - p))
    return None


P_init = rng.normal(0.0, 0.1, size=(n_readers_toy, 2))   # 2 latent factors for the toy run
Q_init = rng.normal(0.0, 0.1, size=(n_books_toy, 2))
neg_per_positive = None   # TODO: n_negatives (how many negatives you sampled per positive)
loss_at_init = None       # TODO: mean_logistic_loss(P_init, Q_init, u_idx, i_idx, y)

## Exercise 3 - One full-batch, per-entity-averaged gradient-descent step

To *fit* the factors we descend the logistic loss. For a pair with score $s = P_u \cdot Q_i$ the gradient of its loss with respect to $s$ is $\sigma(s) - y$; by the chain rule the pair's contribution to $\nabla_{P_u}$ is $(\sigma(s) - y)\,Q_i$ and to $\nabla_{Q_i}$ is $(\sigma(s) - y)\,P_u$. We take a **full-batch** step: sum every pair's contribution per entity, divide each entity's sum by its **own interaction count** (so a reader with many pairs is not pushed harder than a quiet one -- the *per-entity averaging*), add the L2 penalty $\mathrm{reg}\cdot P$, and step **downhill**. Writing $e = y - \sigma(s)$ (the negative of the score gradient), the update reads $P \mathrel{+}= \eta\,(\mathrm{grad}_P / \mathrm{count}_u - \mathrm{reg}\cdot P)$.

Re-initialize $P, Q$ at the same seed, compute the per-entity gradients with the given `accumulate` helper, divide by each entity's interaction count (`np.bincount`, floored at 1), take one step, and confirm the loss **after** the step is below the loss before it. Record both losses and the bool that the step reduced the loss.

In [ ]:
learning_rate = PINNED["learning_rate"]
reg = PINNED["reg"]

rng_step = np.random.default_rng(SEED)
P = rng_step.normal(0.0, 0.1, size=(n_readers_toy, 2))
Q = rng_step.normal(0.0, 0.1, size=(n_books_toy, 2))

loss_before = None        # TODO: mean_logistic_loss(P, Q, u_idx, i_idx, y)

# Per-entity interaction counts (floored at 1 so an unseen entity never divides by zero).
count_u = None            # TODO: np.maximum(np.bincount(u_idx, minlength=n_readers_toy), 1)[:, None]
count_i = None            # TODO: np.maximum(np.bincount(i_idx, minlength=n_books_toy), 1)[:, None]

scores = None             # TODO: (P[u_idx] * Q[i_idx]).sum(axis=1)
err = None                # TODO: y - sigmoid(scores)   -- the negative score-gradient per pair
grad_p = None             # TODO: accumulate(u_idx, err[:, None] * Q[i_idx], n_readers_toy)
grad_q = None             # TODO: accumulate(i_idx, err[:, None] * P[u_idx], n_books_toy)
# TODO: one downhill step with L2 regularization (in place):
#   P += learning_rate * (grad_p / count_u - reg * P)
#   Q += learning_rate * (grad_q / count_i - reg * Q)

loss_after = None         # TODO: mean_logistic_loss(P, Q, u_idx, i_idx, y) AFTER the step
step_reduced_loss = None  # TODO: bool - loss_after < loss_before

## Exercise 4 - Train the toy factors and watch the loss fall

One step helps; **gradient descent** repeats it. Re-initialize $P, Q$ and run the exact step from Exercise 3 for `n_epochs` epochs (use `PINNED["n_epochs"]`), recording the loss every 30 epochs into a list. The toy log has two clean taste clusters, so the loss should fall steeply and then flatten as $\sigma(P_u \cdot Q_i)$ saturates toward $1$ on the positives. Record the loss curve, confirm the final loss is well below the first recorded loss, and -- as the honest read -- confirm the curve is (weakly) **monotone non-increasing** from one recorded point to the next. In one or two sentences, relate `reg` to this curve: why a nonzero L2 penalty keeps the factors from growing without bound even as the loss falls.

In [ ]:
n_epochs = PINNED["n_epochs"]

rng_fit = np.random.default_rng(SEED)
P = rng_fit.normal(0.0, 0.1, size=(n_readers_toy, 2))
Q = rng_fit.normal(0.0, 0.1, size=(n_books_toy, 2))

count_u = np.maximum(np.bincount(u_idx, minlength=n_readers_toy), 1)[:, None]
count_i = np.maximum(np.bincount(i_idx, minlength=n_books_toy), 1)[:, None]

loss_curve = []           # TODO: the recorded losses, one every 30 epochs
for epoch in range(n_epochs):
    pass                  # TODO: repeat the Exercise-3 step on P, Q (scores, err, grad_p, grad_q,
                          #       the per-entity-averaged L2 update); every 30 epochs append
                          #       mean_logistic_loss(P, Q, u_idx, i_idx, y) to loss_curve

first_recorded_loss = None  # TODO: loss_curve[0]
final_loss = None           # TODO: loss_curve[-1]
loss_fell = None            # TODO: bool - final_loss < first_recorded_loss
is_monotone = None          # TODO: bool - every recorded loss is <= the one before it
reg_explanation = "TODO"    # 1-2 sentences: how the L2 penalty (reg) bounds factor growth

## Exercise 5 - Reveal `MatrixFactorizationPath`: fit the real log, and the retrieve contract

The package ships exactly this training as a standard retrieval path. `bookrec.MatrixFactorizationPath(**PINNED)` learns $P, Q$ from the **train positives** of the real log by full-batch, per-entity-averaged gradient descent -- `fit(interactions, catalog=None)` duck-types the row-mapping log (no pandas inside the package) and takes `catalog` as the item universe, so **every** catalog book gets a $Q$ row (MF can *score* the whole catalog). `retrieve(query, context, k)` scores a fitted reader's whole catalog by $P[\text{reader}] \cdot Q^{\top}$, excludes `context["seen"]`, and returns the top $k$.

Fit it on `rows` with `catalog=catalog_ids` at the pinned config. Pick a known reader (the first of `mf.artifact()["reader_ids"]`), retrieve its top 5 given its real `seen` history, and retrieve again with an **empty** `seen`. Confirm the known reader still gets 5 recs with empty `seen` -- MF builds its query from the learned factor, not from `seen` (which only *excludes*). Then retrieve for an **unknown** reader id and confirm the answer is `[]`. In two sentences, contrast the two cold cases: a cold *reader* (no learned factor) versus a cold *book* (gets a $Q$ row but only negative-shaped signal).

In [ ]:
mf = None                 # TODO: bookrec.MatrixFactorizationPath(**PINNED).fit(rows, catalog=catalog_ids)
reader_id = None          # TODO: int(mf.artifact()["reader_ids"][0]) -- a reader with a learned factor
seen = None               # TODO: seen_by[reader_id] -- that reader's train-positive history

top5 = None               # TODO: mf.retrieve(reader_id, {"seen": seen}, k=5)
top5_ids = None           # TODO: [c.item_id for c in top5]

empty_seen = None         # TODO: mf.retrieve(reader_id, {"seen": set()}, k=5)
known_reader_still_recommends = None  # TODO: bool - len(empty_seen) == 5 (empty seen still returns k)

unknown = None            # TODO: mf.retrieve(10_000_000, {"seen": set()}, k=5)
unknown_is_empty = None   # TODO: bool - unknown == []
cold_cases_explanation = "TODO"  # 2 sentences: cold READER (no P row -> []) vs cold BOOK (has a Q
                                 # row but only negative-shaped signal -> scored low; Unit 9 fixes it)

## Exercise 6 - Register the paths and read the validation scoreboard honestly

Line MF up against the earlier units under the Unit 1 rules unchanged. Build a `bookrec.RandomRetrievalPath(catalog_ids, seed=SEED)` floor, fit a `bookrec.PopularityRetrievalPath()`, a `bookrec.LexicalRetrievalPath(keywords)`, and an item-item `bookrec.ItemItemRetrievalPath()` on `rows` with `catalog=catalog_ids`, and register all of them plus the `mf` from Exercise 5 in a fresh `bookrec.PathRegistry`. Score each with `bookrec.run_validation_scoreboard(path, interactions_path, catalog_ids=catalog_ids, k=10, cold_readers=cold_readers)` and record the `hit_rate_at_k`, plus MF's ratios over popularity and the random floor.

Then write two or three **honest** sentences. On this data MF lands **on par with item-item CF** (about 0.25-0.26 vs CF's ~0.25, within noise over 500 readers -- a 0.24x draw would *not* contradict this), while beating the lexical content path and popularity by wide margins. Do **not** write that MF "beats" CF: it is the *latent generalization* of co-occurrence, compressing the same who-read-what signal into compact factors. Say what the ratios do and do not establish.

In [ ]:
random_floor = None       # TODO: bookrec.RandomRetrievalPath(catalog_ids, seed=SEED)
popularity = None         # TODO: bookrec.PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = None            # TODO: bookrec.LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
cf = None                 # TODO: bookrec.ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
registry = None           # TODO: bookrec.PathRegistry(); register random_floor, popularity,
                          #       lexical, cf, and mf (from Exercise 5)


def score(path):
    # TODO: return bookrec.run_validation_scoreboard(
    #           path, interactions_path, catalog_ids=catalog_ids,
    #           k=10, cold_readers=cold_readers).hit_rate_at_k
    return None


mf_hit = None             # TODO: score(mf)
cf_hit = None             # TODO: score(cf)
lexical_hit = None        # TODO: score(lexical)
popularity_hit = None     # TODO: score(popularity)
floor_hit = None          # TODO: score(random_floor)
mf_vs_popularity = None   # TODO: mf_hit / popularity_hit   (expect ~2.3)
mf_vs_floor = None        # TODO: mf_hit / floor_hit        (expect ~21)
scoreboard_reading = "TODO"  # 2-3 honest sentences: MF is ON PAR with CF (not "beats"), within
                             # noise; it beats lexical and popularity by wide margins; the latent
                             # generalization of co-occurrence

## Exercise 7 - Challenge: how many latent factors? (capacity and the exec budget)

`n_factors` is the model's **capacity** -- the number of taste dials. Too few and the factors cannot separate the tastes (underfitting); the shipped default is 32. Sweep a **small grid** of `n_factors` values and watch the val hit@10. To respect the per-cell exec budget (each fit at the pinned 300 epochs is ~16-21 s, and six fits in one cell would brush the 120 s cap), keep the sweep to **at most 4 fits in this cell** -- here three values. Fit `MatrixFactorizationPath` at each `n_factors` (every other setting pinned, `seed=SEED`), score each on `val` with your Exercise-6 `score` helper, and record the `{n_factors: hit}` map. Expect very small factor counts to underfit and 32 to be at or near the best. In one or two sentences, connect factor count to the under/overfitting trade-off (and why L2 `reg` matters more as capacity grows).

In [ ]:
# At most 4 fits in this cell (budget): three factor counts, everything else pinned.
factor_grid = [2, 8, 32]
hit_by_factors = {}       # TODO: for n in factor_grid, fit
                          #   bookrec.MatrixFactorizationPath(
                          #       n_factors=n, n_epochs=PINNED["n_epochs"],
                          #       learning_rate=PINNED["learning_rate"], reg=PINNED["reg"],
                          #       n_negatives=PINNED["n_negatives"], seed=SEED,
                          #   ).fit(rows, catalog=catalog_ids)
                          #   then hit_by_factors[n] = score(that path)
smallest_underfits = None  # TODO: bool - hit_by_factors[2] < hit_by_factors[32]
capacity_explanation = "TODO"  # 1-2 sentences: capacity vs under/overfitting; why reg matters
                               # more as n_factors grows

## Exercise 8 - Challenge: the score IS an embedding dot product (the bridge to Unit 8)

The hinge of Part 1. MF's learned factors are **embeddings**: a reader embedding $P_u$, a book embedding $Q_i$, and the recommendation score is nothing but their **dot product** -- the exact shape Unit 8's neural two-tower will learn (with a network in place of gradient descent on a table). Prove it. Take the fitted `mf` from Exercise 5, pull its `reader_factors` ($P$) and `item_factors` ($Q$), and for the Exercise-5 `reader_id` compute the by-hand score vector $P[\text{row}] \cdot Q^{\top}$ over the whole catalog yourself. Confirm the score `mf.retrieve` reports for each returned book equals your by-hand dot product to floating-point precision -- the retriever is *literally* ranking embedding dot products.

Then compare recommenders: retrieve the same reader's top 5 from the Exercise-6 `cf` (item-item CF) and count how many of MF's top-5 books also appear in CF's top-5. In two or three sentences, interpret the overlap: MF and CF read the *same* co-occurrence signal, so they often agree, but MF generalizes it through the latent factors rather than reading a memorized similarity table -- which is why their picks are close but not identical.

In [ ]:
P = mf.reader_factors          # the learned reader embeddings (readers x factors)
Q = mf.item_factors            # the learned book embeddings (books x factors)
mf_item_ids = mf.artifact()["item_ids"]   # == catalog_ids: the column order of Q
reader_row = {r: idx for idx, r in enumerate(mf.artifact()["reader_ids"])}
row = reader_row[reader_id]    # reader_id and seen come from Exercise 5
col_of_item = {item_id: col for col, item_id in enumerate(mf_item_ids)}

by_hand_scores = None     # TODO: P[row] @ Q.T  -- your own score for every book in mf_item_ids
retrieved = None          # TODO: mf.retrieve(reader_id, {"seen": seen}, k=5)
retrieved_scores = None   # TODO: [c.score for c in retrieved]
by_hand_match = None      # TODO: bool - np.allclose(retrieved_scores,
                          #       [by_hand_scores[col_of_item[c.item_id]] for c in retrieved])

cf_top5 = None            # TODO: cf.retrieve(reader_id, {"seen": seen}, k=5)  (cf from Exercise 6)
cf_top5_ids = None        # TODO: [c.item_id for c in cf_top5]
mf_top5_ids = None        # TODO: [c.item_id for c in retrieved]
overlap_count = None      # TODO: int - how many of mf_top5_ids appear in cf_top5_ids
bridge_explanation = "TODO"  # 2-3 sentences: the score is an embedding dot product (-> Unit 8's
                             # two-tower); MF and CF read the same co-occurrence signal so they
                             # overlap, but MF generalizes via latent factors vs CF's memorized table